# Snowball Strategy

In [ ]:
from datetime import UTC, datetime, timedelta
from itertools import pairwise
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from aws import AthenaDataSource
from core import (
    BacktestTaskDefinition,
    CandleGranularity,
    CsvResultStore,
    CurrencyPair,
    InMemoryTaskRegistry,
    LogLevel,
    StrategyParameters,
    TaskManager,
    TaskProfilingConfig,
    TaskResultRecorder,
    TickGranularity,
    TickGranularityFilter,
    TqdmProgressReporter,
    configure_logging,
)
from IPython.display import display
from matplotlib.patches import Rectangle
from snowball import SnowballStrategy

configure_logging(level=LogLevel.WARNING)


## Parameters

In [ ]:
INSTRUMENT = CurrencyPair.of("USD_JPY")
START_AT = datetime(2026, 1, 1, tzinfo=UTC)
END_AT = datetime(2026, 1, 31, 23, 59, 59, 999999, tzinfo=UTC)
SAMPLE_GRANULARITY = TickGranularity.TICK
CHART_DAY_AGGS_THRESHOLD_DAYS = 31
PROFILE_DIR = Path("profiles") / "snowball"
RESULT_DIR = Path("results") / "snowball"
LOCAL_TIMEZONE = datetime.now().astimezone().tzinfo

data_source = AthenaDataSource.from_env().with_filters(
    TickGranularityFilter.of(SAMPLE_GRANULARITY),
)

parameters = SnowballStrategy.normalize_parameters(
    StrategyParameters.of(
        sizing={
            "base_units": "1000",
        },
        grid={
            "max_retracements_per_layer": 10,
            "max_layers": 10,
            "refill": {
                "enabled": True,
            },
        },
        cycle={
            "hedging_enabled": True,
            "reseed_when_all_positions_pending_rebuild": True,
        },
        forward={
            "take_profit_pips": "20",
        },
        counter={
            "interval": {
                # choices: "constant", "additive", "subtractive"
                #          "multiplicative", "divisive", "manual"
                "mode": "divisive",
                "head_pips": "30",
                "tail_pips": "10",
                "gamma": "1.3",
            },
            "take_profit": {"mode": "weighted_avg"},
        },
        stop_loss={
            "enabled": True,
            # choices: "auto", "distance"
            "mode": "auto",
        },
        rebuild={
            "enabled": True,
            "price": {
                # choices: "original_entry_price", "stop_loss_exit_price"
                "entry_price_mode": "stop_loss_exit_price",
            },
            "stop_loss": {
                # choices: "same_price", "same_distance", "manual_distance"
                "mode": "same_distance",
            },
            "take_profit": {
                # choices: "same_price", "same_distance", "progressive_distance"
                "mode": "same_distance"
            },
        },
        protection={
            "shrink_enabled": False,
            "shrink_start_margin_percent": "70",
            "shrink_target_margin_percent": "50",
            "emergency_enabled": False,
            "emergency_margin_percent": "95",
        },
        account={
            "currency": "USD",
            "balance": {
                "amount": "10000",
                "currency": "USD",
            },
            "margin_rate": "0.04",
            "quote_to_account_rate": "1",
        },
    )
)


## Executuin

In [ ]:
definition = BacktestTaskDefinition(
    name=f"{INSTRUMENT} Snowball backtest",
    instrument=INSTRUMENT,
    start_at=START_AT,
    end_at=END_AT,
    parameters=parameters,
)

strategy = SnowballStrategy(
    name="snowball",
    parameters=definition.parameters,
)

recorder = TaskResultRecorder(
    stores=(CsvResultStore(RESULT_DIR),),
    metric_interval=timedelta(minutes=5),
)

with TaskManager(
    registry=InMemoryTaskRegistry(),
    observers=(recorder,),
    event_handlers=(recorder,),
    profiling=TaskProfilingConfig.for_backtest_period(
        start_at=START_AT,
        end_at=END_AT,
        cprofile=False,
        cprofile_output_path=PROFILE_DIR,
        pyinstrument=False,
        pyinstrument_output_path=PROFILE_DIR,
    ),
) as manager:
    run = manager.start_backtest(definition, data_source=data_source, strategy=strategy)
    final_task = run.wait(progress=TqdmProgressReporter())

print(f"Task {final_task.id} finished with status: {final_task.status.value}")
if final_task.status.value == "failed":
    if final_task.failure is None:
        raise RuntimeError("Task failed without failure details. Restart the kernel and rerun this cell.")
    print(f"Failure: {final_task.failure}")
    if final_task.failure.cause_type:
        print(f"Cause: {final_task.failure.cause_type}")
    if final_task.failure.traceback:
        print(final_task.failure.traceback)
    raise RuntimeError(str(final_task.failure))

print(f"Events recorded: {len(recorder.event_records(final_task.id))}")
print(f"Trades recorded: {len(recorder.trade_summaries(final_task.id))}")


## Results

In [ ]:
events = recorder.event_records(final_task.id)
trades = recorder.trade_summaries(final_task.id)
cycles = recorder.cycle_summaries(final_task.id)
task_summary = recorder.task_summary(final_task)
metrics = tuple(metric for metric in recorder.memory.metrics if metric.task_id == final_task.id)


def local_datetime(value):
    return value.astimezone(LOCAL_TIMEZONE) if value is not None else None


def text(value):
    return str(value) if value is not None else None


signals_df = pd.DataFrame(
    {
        "timestamp": local_datetime(event.timestamp),
        "action": event.action.value,
        "display_id": event.display_id,
        "units": text(event.units),
        "price": text(event.price),
        "rule": event.rule,
        "cycle_id": event.cycle_id,
        "direction": event.direction,
        "entry_role": event.entry_role,
        "layer_number": event.layer_number,
        "slot_number": event.slot_number,
        "build_number": event.build_number,
        "close_reason": event.close_reason,
        "is_rebuild": event.is_rebuild,
        "planned_entry_price": text(event.planned_entry_price),
        "filled_entry_price": text(event.filled_entry_price),
        "planned_take_profit_price": text(event.planned_take_profit_price),
        "filled_take_profit_price": text(event.filled_take_profit_price),
        "planned_stop_loss_price": text(event.planned_stop_loss_price),
        "filled_stop_loss_price": text(event.filled_stop_loss_price),
        "planned_rebuild_price": text(event.planned_rebuild_price),
        "filled_rebuild_price": text(event.filled_rebuild_price),
        "realized_pl": text(event.realized_pl),
    }
    for event in events
)

trade_summary_df = pd.DataFrame(
    {
        "trade_id": trade.trade_id,
        "display_id": trade.display_id,
        "cycle_id": trade.cycle_id,
        "direction": trade.direction,
        "entry_role": trade.entry_role,
        "layer_number": trade.layer_number,
        "slot_number": trade.slot_number,
        "build_number": trade.build_number,
        "opened_at": local_datetime(trade.opened_at),
        "closed_at": local_datetime(trade.closed_at),
        "close_reason": trade.close_reason,
        "units": text(trade.units),
        "filled_entry_price": text(trade.filled_entry_price),
        "filled_take_profit_price": text(trade.filled_take_profit_price),
        "filled_stop_loss_price": text(trade.filled_stop_loss_price),
        "realized_pl": text(trade.realized_pl),
    }
    for trade in trades
)

cycle_summary_df = pd.DataFrame(
    {
        "cycle_id": cycle.cycle_id,
        "trade_ids": list(cycle.trade_ids),
        "opened_at": local_datetime(cycle.opened_at),
        "closed_at": local_datetime(cycle.closed_at),
        "trade_count": cycle.trade_count,
        "open_trade_count": cycle.open_trade_count,
        "closed_trade_count": cycle.closed_trade_count,
        "realized_pl": text(cycle.realized_pl),
    }
    for cycle in cycles
)

task_summary_df = pd.DataFrame(
    []
    if task_summary is None
    else [
        {
            "task_id": task_summary.task_id,
            "task_name": task_summary.task_name,
            "status": task_summary.status,
            "started_at": local_datetime(task_summary.started_at),
            "finished_at": local_datetime(task_summary.finished_at),
            "trade_count": task_summary.trade_count,
            "open_trade_count": task_summary.open_trade_count,
            "closed_trade_count": task_summary.closed_trade_count,
            "realized_pl": text(task_summary.realized_pl),
        }
    ]
)

metrics_df = pd.DataFrame(
    {
        "timestamp": local_datetime(metric.timestamp),
        "realized_pl": text(metric.realized_pl),
        "unrealized_pl": text(metric.unrealized_pl),
        "total_pl": text(metric.total_pl),
        "open_trade_count": metric.open_trade_count,
        "closed_trade_count": metric.closed_trade_count,
    }
    for metric in metrics
)

events_df = signals_df

display(signals_df)
display(trade_summary_df)
display(cycle_summary_df)
display(task_summary_df)
metrics_df


## Chart


In [ ]:
chart_granularity = (
    CandleGranularity.DAY
    if END_AT - START_AT > timedelta(days=CHART_DAY_AGGS_THRESHOLD_DAYS)
    else CandleGranularity.MINUTE_1
)

candles = tuple(
    data_source.candles(
        instrument=INSTRUMENT,
        granularity=chart_granularity,
        start_at=START_AT,
        end_at=END_AT,
    )
)

candles_df = pd.DataFrame(
    {
        "timestamp": local_datetime(candle.timestamp),
        "open": float(candle.open.amount),
        "high": float(candle.high.amount),
        "low": float(candle.low.amount),
        "close": float(candle.close.amount),
        "volume": candle.volume,
    }
    for candle in candles
)

fig, ax = plt.subplots(figsize=(16, 8))

if not candles_df.empty:
    x_values = mdates.date2num(candles_df["timestamp"])
    positive_deltas = [right - left for left, right in pairwise(x_values) if right > left]
    width = min(positive_deltas) * 0.8 if positive_deltas else 0.7
    if chart_granularity == CandleGranularity.MINUTE_1:
        width = min(width, 1 / 24 / 60 * 0.8)

    for x, row in zip(x_values, candles_df.itertuples(index=False), strict=False):
        color = "#16833a" if row.close >= row.open else "#b42318"
        ax.vlines(x, row.low, row.high, color=color, linewidth=0.8, alpha=0.9)
        lower = min(row.open, row.close)
        height = abs(row.close - row.open)
        if height == 0:
            ax.hlines(row.open, x - width / 2, x + width / 2, color=color, linewidth=1.2)
        else:
            ax.add_patch(
                Rectangle(
                    (x - width / 2, lower),
                    width,
                    height,
                    facecolor=color,
                    edgecolor=color,
                    linewidth=0.6,
                    alpha=0.75,
                )
            )


def event_points(predicate):
    return [
        (local_datetime(event.timestamp), float(event.price.amount))
        for event in events
        if event.price is not None and predicate(event)
    ]


def scatter(points, *, label, marker, color):
    if not points:
        return
    timestamps, prices = zip(*points, strict=False)
    ax.scatter(timestamps, prices, label=label, marker=marker, color=color, s=48, zorder=5)


scatter(
    event_points(
        lambda event: event.action.value == "open_trade" and not event.is_rebuild
    ),
    label="open",
    marker="^",
    color="#2563eb",
)
scatter(
    event_points(
        lambda event: (
            event.action.value == "close_trade"
            and event.close_reason != "stop_loss"
        )
    ),
    label="close",
    marker="v",
    color="#0f766e",
)
scatter(
    event_points(
        lambda event: event.action.value == "open_trade" and event.is_rebuild
    ),
    label="rebuild",
    marker="D",
    color="#9333ea",
)
scatter(
    event_points(lambda event: event.close_reason == "stop_loss"),
    label="stop_loss",
    marker="x",
    color="#dc2626",
)

ax.set_title(f"{INSTRUMENT} {chart_granularity.value} Snowball")
ax.set_ylabel(str(INSTRUMENT.quote))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M", tz=LOCAL_TIMEZONE))
ax.grid(True, axis="y", alpha=0.25)
handles, labels = ax.get_legend_handles_labels()
if handles:
    ax.legend(handles, labels, loc="best")
fig.autofmt_xdate()
plt.show()


In [ ]:
profile = run.profile()
profile_outputs = {
    "cprofile": profile.cprofile_output_path,
    "pyinstrument": profile.pyinstrument_output_path,
}
if profile.cprofile_output_path is not None:
    profile_outputs["snakeviz"] = f"uv run snakeviz {profile.cprofile_output_path}"
    profile_outputs["tuna"] = f"uv run tuna {profile.cprofile_output_path}"
if profile.pyinstrument_output_path is not None:
    profile_outputs["html"] = f"open {profile.pyinstrument_output_path}"
profile_outputs